# Demo 09f — Use cases: one task, three methods

**Class 09 · E3** (slides: `slides/09_blind_signal_separation.md`, block "Use Cases")

PCA, ICA and NMF all write X ≈ W H, with a different constraint each. Here the *same* task
is given to all three, with the same metric, and the verdict is asserted: some tasks
are shared (compression, features, sound), some belong to one method (topics, soft
clusters, strokes and recommendations to NMF; blink removal to ICA; anomaly scores to PCA).
ICA is run as a rotation of the PCA subspace (`bss.ica_in_pca_space`): same subspace, new basis.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import pynnmf
from matplotlib.colors import ListedColormap
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA, FastICA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import adjusted_rand_score, matthews_corrcoef, roc_auc_score

from faa import bss, data, plotlib

warnings.filterwarnings("ignore")
pl.Config.set_tbl_rows(40)
pl.Config.set_tbl_cols(20)
pl.Config.set_float_precision(3)
pl.Config.set_tbl_hide_dataframe_shape(True)
pl.Config.set_tbl_formatting("ASCII_MARKDOWN")
pl.Config.set_tbl_width_chars(150)
plotlib.setup(module="09")
BLUE, ORANGE, GREEN, RED, SKY, PURPLE, GREY = (
    plotlib.BLUE,
    plotlib.ORANGE,
    plotlib.GREEN,
    plotlib.RED,
    plotlib.SKY,
    plotlib.PURPLE,
    plotlib.GREY,
)
SEED = 0
RESULTS: dict[str, tuple[float, float, float]] = {}  # use case -> (PCA, ICA, NMF) key metric, filled below


def slide(title: str) -> None:
    """Every cell that prints a number of a slide starts with this line."""
    print(f"--- slide: {title} ---")


def rel_err(X: np.ndarray, R: np.ndarray) -> float:
    return float(np.linalg.norm(X - R) / np.linalg.norm(X))


Xd, yd, _ = data.load_table("digits", "target")
Xd = Xd / Xd.max()
rs = np.random.default_rng(1)
idx = rs.permutation(len(Xd))
tr, te = idx[:1000], idx[1000:1400]

## Slide: Sound — three notes on a spectrogram

Three harmonic tones, each on and off in its own pattern, summed with noise. The magnitude
spectrogram V (frequency x time) is non-negative: NMF gives W (a spectrum per note) and H
(when each note plays). PCA and ICA see time frames as samples: their "spectra" are signed.
Score: correlation of each recovered activation with the true one, and of each recovered
spectrum with the true spectrum (best match, sign fixed).

In [ ]:
V, A_true, T_true = bss.spectro_mix(SEED, notes=3)
c = 3
_, Wn, Hn, _ = pynnmf.nmf_hals(V, c, n=400, seed=SEED, init="nndsvd")  # Wn (freq, c), Hn (c, time)
pc = PCA(c).fit(V.T)
ic = FastICA(c, random_state=SEED, max_iter=2000).fit(V.T)
acts = {"PCA": pc.transform(V.T).T, "ICA": ic.transform(V.T).T, "NMF": Hn}
temps = {"PCA": pc.components_, "ICA": ic.mixing_.T, "NMF": Wn.T}
rows = []
for m in ("PCA", "ICA", "NMF"):
    ma, ca = bss.match(A_true, acts[m])
    mt, ct = bss.match(T_true, temps[m])
    neg = float(np.abs(np.minimum(temps[m], 0)).sum() / np.abs(temps[m]).sum())
    rows.append(
        {"method": m, "activation corr": ca.mean(), "spectrum corr": ct.mean(), "negative mass of spectra": neg}
    )
slide("Sound: three notes on a spectrogram (c = 3)")
snd = pl.DataFrame(rows)
print(snd.with_columns(pl.col(pl.Float64).round(3)))
RESULTS["sound: spectrum corr"] = tuple(snd["spectrum corr"].to_list())  # type: ignore
assert snd["spectrum corr"][2] > max(snd["spectrum corr"][0], snd["spectrum corr"][1]), (
    "NMF must recover the spectra best"
)
assert snd["negative mass of spectra"][2] == 0 and snd["negative mass of spectra"][0] > 0.1

f, ax = plt.subplots(1, 4, figsize=(plotlib.FULL, 1.25), sharey=True)
panels = [("true", A_true)] + [(m, bss.match(A_true, acts[m])[0]) for m in ("PCA", "ICA", "NMF")]
for a, (name, M) in zip(ax, panels):
    for k, col in enumerate((BLUE, ORANGE, GREEN)):
        a.plot(M[k] / np.abs(M[k]).max(), color=col, lw=0.9)
    a.set_title(name, fontsize=7.5)
    a.set_xticks([])
ax[0].set_ylabel("activation")
plotlib.save(f, "p6_spectro")

## Slide: Image parts — strokes of a digit

NMF with c = 8 parts on the digits. For each ink pixel of an image, the part with the largest
contribution W[i,j] H[j,p] names its segment: the digit is cut into strokes, with no labels.
PCA and ICA components are signed blurs (no segment boundaries). Score: Hoyer sparseness of
the components (0 = every pixel used, 1 = one pixel).

In [ ]:
c = 8
_, Wd, Hd, _ = pynnmf.nmf_hals(Xd, c, n=400, seed=SEED, init="nndsvd")
pcd, icd = bss.ica_in_pca_space(Xd, c, SEED)
comps = {"PCA": pcd.components_, "ICA": bss.ica_components(pcd, icd), "NMF": Hd}
slide("Image parts: sparseness of the components (c = 8)")
hs = {m: float(np.mean([bss.hoyer(v) for v in C])) for m, C in comps.items()}
print({m: round(v, 3) for m, v in hs.items()})
RESULTS["strokes: Hoyer sparseness"] = (hs["PCA"], hs["ICA"], hs["NMF"])
assert hs["NMF"] > max(hs["PCA"], hs["ICA"]) + 0.15
neg = {m: round(float(np.abs(np.minimum(C, 0)).sum() / np.abs(C).sum()), 3) for m, C in comps.items()}
print("negative mass of the components:", neg)

shown = [np.flatnonzero(yd == k)[0] for k in (0, 2, 3, 5, 7, 9)]
cols8 = ListedColormap(
    [plotlib.BLUE, plotlib.ORANGE, plotlib.GREEN, plotlib.RED, plotlib.SKY, plotlib.PURPLE, "#F0E442", "#555555"]
)
f, ax = plt.subplots(2, len(shown), figsize=(plotlib.FULL, 1.45))
nparts = []
for j, i in enumerate(shown):
    seg = np.argmax(Wd[i][:, None] * Hd, axis=0).astype(float)
    ink = Xd[i] > 0.2
    seg[~ink] = np.nan
    nparts.append(len(np.unique(seg[ink])))
    ax[0, j].imshow(Xd[i].reshape(8, 8), cmap="gray_r")
    ax[1, j].imshow(seg.reshape(8, 8), cmap=cols8, vmin=0, vmax=7)
    for r in (0, 1):
        ax[r, j].set_xticks([]), ax[r, j].set_yticks([])
ax[0, 0].set_ylabel("digit", fontsize=7)
ax[1, 0].set_ylabel("strokes", fontsize=7)
plotlib.save(f, "p6_segments")
print(f"parts used by the six shown digits: {nparts} (of {c})")

## Slide: Compression and denoising — the job of PCA

Fit on clean training digits, project *noisy* test digits (noise 0.3, clipped at 0) on c
components and reconstruct. PCA is the minimum-error subspace. ICA uses the same subspace
with a rotated basis (identical error). NMF pays for non-negativity.

In [ ]:
noise = np.clip(Xd[te] + rs.normal(0, 0.3, Xd[te].shape), 0, None)
rows = []
for c in (2, 4, 8, 16, 32):
    p, i = bss.ica_in_pca_space(Xd[tr], c, SEED)
    rp = p.inverse_transform(p.transform(noise))
    ri = p.inverse_transform(i.inverse_transform(i.transform(p.transform(noise))))
    _, _, Hc, _ = pynnmf.nmf_hals(Xd[tr], c, n=200, seed=SEED, init="nndsvd")
    rn = bss.nnls_scores(noise, Hc) @ Hc
    rows.append(
        {
            "c": c,
            "PCA": rel_err(Xd[te], rp),
            "ICA": rel_err(Xd[te], ri),
            "NMF": rel_err(Xd[te], rn),
            "noisy input": rel_err(Xd[te], noise),
        }
    )
    assert np.allclose(rp, ri, atol=1e-8), "ICA reconstruction must equal PCA"
slide("Denoising: relative error of the reconstruction (noisy test digits)")
den = pl.DataFrame(rows)
print(den.with_columns(pl.col(pl.Float64).round(3)))
r16 = den.filter(pl.col("c") == 16).row(0, named=True)
RESULTS["denoise c=16: 1 - rel. error"] = (1 - r16["PCA"], 1 - r16["ICA"], 1 - r16["NMF"])
assert r16["PCA"] < r16["NMF"] < r16["noisy input"]

f, ax = plt.subplots(figsize=(plotlib.COL, 1.45))
ax.plot(den["c"], den["PCA"], color=BLUE, lw=1.4, label="PCA")
ax.plot(den["c"], den["ICA"], color=GREEN, lw=1.0, ls="--", label="ICA (same)")
ax.plot(den["c"], den["NMF"], color=ORANGE, lw=1.4, label="NMF")
ax.axhline(den["noisy input"][0], color=GREY, lw=0.8, ls=":")
ax.text(32, den["noisy input"][0] + 0.004, "noisy input", ha="right", fontsize=6, color=GREY)
ax.set_xscale("log", base=2), ax.set_xticks([2, 4, 8, 16, 32]), ax.set_xticklabels([2, 4, 8, 16, 32])
ax.set_xlabel("components c"), ax.set_ylabel("relative error")
ax.set_ylim(0.28, 0.54)
ax.legend(fontsize=6, loc="lower left", ncol=3)
plotlib.save(f, "p6_denoise")

## Slide: Features before a classifier

The c scores of each method feed a logistic regression on the digits (the Class 03 model).
Score: MCC on held-out digits. Supervised-blind compression: none of the three sees the labels.

In [ ]:
rows = []
for c in (2, 4, 8, 16):
    p, i = bss.ica_in_pca_space(Xd[tr], c, SEED)
    _, _, Hc, _ = pynnmf.nmf_hals(Xd[tr], c, n=200, seed=SEED, init="nndsvd")
    feats = {
        "PCA": lambda X: p.transform(X),
        "ICA": lambda X: i.transform(p.transform(X)),
        "NMF": lambda X: bss.nnls_scores(X, Hc),
    }
    row = {"c": c}
    for m, fn in feats.items():
        clf = LogisticRegression(max_iter=3000).fit(fn(Xd[tr]), yd[tr])
        row[m] = matthews_corrcoef(yd[te], clf.predict(fn(Xd[te])))
    rows.append(row)
slide("Features before a classifier: MCC of logistic regression")
mcc = pl.DataFrame(rows)
print(mcc.with_columns(pl.col(pl.Float64).round(3)))
m16 = mcc.filter(pl.col("c") == 16).row(0, named=True)
RESULTS["features c=16: MCC"] = (m16["PCA"], m16["ICA"], m16["NMF"])
assert min(m16["PCA"], m16["ICA"], m16["NMF"]) > 0.8, "all three are usable features at c = 16"

## Slide: Topics — NMF on word frequencies

Spambase rows are e-mails, columns word frequencies (non-negative). NMF with 5 topics:
each row of H is a list of words with non-negative weights, each e-mail a mixture of topics.
PCA components on the same data carry both signs: their "topics" are contrasts, not lists.

In [ ]:
Xs, ys, cols = data.load_table("spambase", "spam")
wi = [j for j, nm in enumerate(cols) if nm.startswith("word_freq_")]
words = [cols[j].removeprefix("word_freq_") for j in wi]
Xw = Xs[:, wi]
_, Ws, Hs, _ = pynnmf.nmf_hals(Xw, 5, n=300, seed=SEED, init="nndsvd")
slide("Topics: top words of five NMF topics on spambase")
for k in range(5):
    top = np.argsort(Hs[k])[::-1][:6]
    spam_share = (
        float(ys[np.argmax(Ws * np.linalg.norm(Hs, axis=1), axis=1) == k].mean())
        if (np.argmax(Ws * np.linalg.norm(Hs, axis=1), axis=1) == k).any()
        else float("nan")
    )
    print(f"topic {k + 1}: {', '.join(words[j] for j in top)}  (spam share of its e-mails {spam_share:.2f})")
pcs = PCA(3).fit(Xw)
for k in range(2):
    o = np.argsort(pcs.components_[k])
    print(f"PC{k + 1}: + {', '.join(words[j] for j in o[::-1][:4])}  |  - {', '.join(words[j] for j in o[:4])}")
neg_pc = float(np.abs(np.minimum(pcs.components_, 0)).sum() / np.abs(pcs.components_).sum())
print(f"negative mass: PCA components {neg_pc:.2f}, NMF topics {float(np.abs(np.minimum(Hs, 0)).sum()):.2f}")
assert Hs.min() >= 0 and neg_pc > 0.1

## Slide: Soft clustering — blended customers

300 customers whose habits are a *blend* of three profiles (memberships sum to one). NMF
returns the memberships W (rescaled to sum to one). k-means returns one label; a soft
k-means score (inverse distance to each centre) is the best a clustering can do. Failure
case: the digit classes are not NMF parts, so an arg-max over parts is a poor classifier.

In [ ]:
Xm, Mtrue, _ = bss.mixed_membership(300, SEED)
_, Wm, Hm, _ = pynnmf.nmf_hals(Xm, 3, n=400, seed=SEED, init="nndsvd")
Wm = Wm * np.linalg.norm(Hm, axis=1)
Pn = Wm / Wm.sum(1, keepdims=True)
km = KMeans(3, n_init=10, random_state=SEED).fit(Xm)
D = km.transform(Xm)
Pk = (1 / D) / (1 / D).sum(1, keepdims=True)
corr = lambda P: float(np.mean(np.abs(np.corrcoef(np.vstack([Mtrue.T, P.T]))[:3, 3:]).max(1)))  # noqa: E731
slide("Soft clustering: correlation of recovered and true memberships")
cn, ck = corr(Pn), corr(Pk)
print(
    f"NMF memberships {cn:.3f}, soft k-means {ck:.3f}; share of customers with no membership above 0.8: {float((Mtrue.max(1) < 0.8).mean()):.2f}"
)
RESULTS["soft clusters: membership corr"] = (float("nan"), float("nan"), cn)
assert cn > ck > 0.5

_, Wc10, Hc10, _ = pynnmf.nmf_hals(Xd, 10, n=300, seed=SEED, init="nndsvd")
ari_n = adjusted_rand_score(yd, Wc10.argmax(1))
ari_k = adjusted_rand_score(yd, KMeans(10, n_init=10, random_state=SEED).fit(Xd).labels_)
slide("Failure: arg-max of NMF parts vs k-means on the digit classes")
print(f"adjusted Rand index: NMF arg-max {ari_n:.3f}, k-means {ari_k:.3f}")
assert ari_k > ari_n

f, ax = plt.subplots(1, 3, figsize=(plotlib.FULL, 1.5), sharey=True)
order = np.argsort(Mtrue.argmax(1) * 10 + Mtrue.max(1))[::15]
for a, (name, M) in zip(ax, (("true", Mtrue), ("NMF", Pn), ("soft k-means", Pk))):
    a.imshow(
        M[order].T
        if name == "true"
        else M[order][:, np.argmax(np.abs(np.corrcoef(np.vstack([Mtrue.T, M.T]))[:3, 3:]), axis=1)].T,
        cmap="viridis",
        aspect="auto",
        vmin=0,
        vmax=1,
    )
    a.set_title(name, fontsize=7.5)
    a.set_xticks([]), a.set_yticks([0, 1, 2]), a.set_yticklabels(["A", "B", "C"], fontsize=6)
ax[0].set_ylabel("profile")
plotlib.save(f, "p6_membership")

## Slide: Artifact removal — an eye blink in five channels

Three brain-like rhythms plus one large, spiky blink, mixed into five channels. ICA finds the
blink as one *independent* source: zero it, remix, done. PCA's first component is the blink
mixed with the rhythms (they share variance), so zeroing it removes brain signal too.
Score: RMSE of the cleaned channels against the blink-free signal (both centred).

In [ ]:
Xe, clean, blink, _ = bss.eeg_like(SEED)
cc = clean - clean.mean(1, keepdims=True)
pe, ie = bss.ica_in_pca_space(Xe.T, 4, SEED)
Z = ie.transform(pe.transform(Xe.T))
kb = int(np.argmax([abs(np.corrcoef(Z[:, j], blink)[0, 1]) for j in range(4)]))
cb = abs(np.corrcoef(Z[:, kb], blink)[0, 1])
Zc = Z.copy()
Zc[:, kb] = 0
x_ica = pe.inverse_transform(ie.inverse_transform(Zc)).T
pp = PCA(4).fit(Xe.T)
P = pp.transform(Xe.T)
kp = int(np.argmax([abs(np.corrcoef(P[:, j], blink)[0, 1]) for j in range(4)]))
cp = abs(np.corrcoef(P[:, kp], blink)[0, 1])
Pc = P.copy()
Pc[:, kp] = 0
x_pca = pp.inverse_transform(Pc).T
rm = lambda Y: float(np.sqrt(((Y - Y.mean(1, keepdims=True) - cc) ** 2).mean()))  # noqa: E731
xc = Xe - Xe.mean(1, keepdims=True)
slide("Artifact removal: RMSE against the blink-free signal")
print(f"corr(blink, component): ICA {cb:.3f}, PCA {cp:.3f}")
print(f"RMSE: noisy {rm(xc):.3f}, PCA-cleaned {rm(x_pca):.3f}, ICA-cleaned {rm(x_ica):.3f}")
RESULTS["blink removal: 1 / RMSE"] = (1 / rm(x_pca), 1 / rm(x_ica), float("nan"))
assert cb > 0.99 > cp and rm(x_ica) < 0.5 * rm(x_pca) and rm(x_pca) < rm(xc)

f, ax = plt.subplots(3, 1, figsize=(plotlib.FULL, 1.9), sharex=True, sharey=True)
ch = 4
for a, (name, Y, col) in zip(ax, (("noisy", xc, GREY), ("PCA", x_pca, BLUE), ("ICA", x_ica, GREEN))):
    a.plot(cc[ch], color=LIGHT if (LIGHT := plotlib.LIGHT) else GREY, lw=1.6)
    a.plot(Y[ch], color=col, lw=0.8)
    a.set_ylabel(name, fontsize=6)
    a.set_yticks([])
ax[-1].set_xlabel("time (samples); light line = blink-free truth")
plotlib.save(f, "p6_blink")

## Slide: Anomaly scores — reconstruction error

Twenty test digits get a bright 3x3 block pasted on. The score is the error of the
reconstruction from the c = 8 components fitted on clean digits (ICA = PCA error again).

In [ ]:
Xa = Xd[te].copy()
bad = rs.choice(len(te), 20, replace=False)
for b in bad:
    Xa[b].reshape(8, 8)[2:5, 2:5] = 1.0
lab = np.zeros(len(te))
lab[bad] = 1
p8 = PCA(8).fit(Xd[tr])
sc_p = np.linalg.norm(Xa - p8.inverse_transform(p8.transform(Xa)), axis=1)
_, _, H8, _ = pynnmf.nmf_hals(Xd[tr], 8, n=200, seed=SEED, init="nndsvd")
sc_n = np.linalg.norm(Xa - bss.nnls_scores(Xa, H8) @ H8, axis=1)
auc_p, auc_n = roc_auc_score(lab, sc_p), roc_auc_score(lab, sc_n)
slide("Anomaly detection: AUC of the reconstruction error")
print(f"AUC: PCA (= ICA) {auc_p:.3f}, NMF {auc_n:.3f}")
RESULTS["anomaly: AUC"] = (auc_p, auc_p, auc_n)
f, ax = plt.subplots(1, 2, figsize=(plotlib.FULL, 1.25), sharey=True)
for a_, (name, sc) in zip(ax, (("PCA (= ICA)", sc_p), ("NMF", sc_n))):
    bins = np.linspace(0, sc.max(), 25)
    a_.hist(sc[lab == 0], bins=bins, color=SKY, label="clean")
    a_.hist(sc[lab == 1], bins=bins, color=RED, label="pasted block")
    a_.set_title(name, fontsize=7.5)
    a_.set_xlabel("reconstruction error")
ax[0].set_ylabel("digits")
ax[0].legend(fontsize=6)
plotlib.save(f, "p6_anomaly")
assert auc_p > auc_n > 0.5

## Slide: Imputation — missing pixels, five fillers

A fraction of the pixels of the digits is lost at random. pyNNMF has four solvers that accept NaN
entries (ALS, HALS, MU, rwnmf): they factorise the observed cells only and fill the rest from W H.
PCA needs complete data: iterative PCA fills with the mean, projects, refills the holes, repeats.
Score: RMSE on the hidden pixels (rank 8, same iteration budget); baseline: the pixel mean.

In [ ]:
SOLVERS = {
    "ALS": lambda X, c, init: pynnmf.nmf_als(X, c, n=150, seed=SEED, init=init),
    "HALS": lambda X, c, init: pynnmf.nmf_hals(X, c, n=150, seed=SEED, init=init),
    "MU": lambda X, c, init: pynnmf.nmf_mu(X, c, n=150, seed=SEED, init=init),
    "rwnmf": lambda X, c, init: pynnmf.rwnmf(X, c, num_iter=150, seed=SEED, init=init),
}


def nmf_fill(name: str, Xh: np.ndarray, c: int, init: str) -> np.ndarray:
    """Reconstruction W H of the chosen pyNNMF solver (rwnmf returns V of shape (n, c): transpose it)."""
    out = SOLVERS[name](Xh, c, init)
    return out[1] @ out[2].T if name == "rwnmf" else out[1] @ out[2]


def pca_em(X_true: np.ndarray, observed: np.ndarray, c: int, rounds: int = 40) -> np.ndarray:
    """Iterative PCA imputation: fill with the column mean, project on c PCs, refill the holes, repeat."""
    Xh = np.where(observed, X_true, np.nan)
    Xi = np.where(observed, X_true, np.nanmean(Xh, 0))
    for _ in range(rounds):
        p = PCA(c).fit(Xi)
        Xi = np.where(observed, X_true, p.inverse_transform(p.transform(Xi)))
    return Xi


rows = []
for frac in (0.1, 0.25, 0.5):
    mk = np.random.default_rng(SEED).random(Xd.shape) > frac  # True = observed
    Xh = np.where(mk, Xd, np.nan)
    hid = lambda Y: float(np.sqrt(((np.clip(Y, 0, 1) - Xd)[~mk] ** 2).mean()))  # noqa: E731
    row = {"missing": frac, "mean": hid(np.where(mk, Xd, np.nanmean(Xh, 0))), "PCA (iterative)": hid(pca_em(Xd, mk, 8))}
    for name in SOLVERS:
        row[name] = hid(nmf_fill(name, Xh, 8, "nndsvd"))
    rows.append(row)
slide("Imputation of missing pixels: RMSE on the hidden entries (rank 8)")
imp = pl.DataFrame(rows)
print(imp.with_columns(pl.col(pl.Float64).round(3)))
r25 = imp.filter(pl.col("missing") == 0.25).row(0, named=True)
nmf_best = min(r25[k] for k in SOLVERS)
RESULTS["imputation, dense pixels: 1 / RMSE"] = (1 / r25["PCA (iterative)"], 1 / r25["PCA (iterative)"], 1 / nmf_best)
assert r25["PCA (iterative)"] < nmf_best < r25["mean"], "dense data: iterative PCA is hard to beat"
assert min(r25, key=lambda k: r25[k] if k in SOLVERS else 9) == "HALS", "HALS is the best NMF filler on pixels"

mk = np.random.default_rng(SEED).random(Xd.shape) > 0.25
Xh = np.where(mk, Xd, np.nan)
fills = {
    "lost": np.where(mk, Xd, 0.5),
    "mean": np.where(mk, Xd, np.nanmean(Xh, 0)),
    "PCA": pca_em(Xd, mk, 8),
    "NMF": np.where(mk, Xd, np.clip(nmf_fill("HALS", Xh, 8, "nndsvd"), 0, 1)),
}
f, ax = plt.subplots(4, 6, figsize=(plotlib.FULL, 2.2))
for j, i in enumerate(shown):
    for r, (name, Y) in enumerate(fills.items()):
        ax[r, j].imshow(Y[i].reshape(8, 8), cmap="gray_r", vmin=0, vmax=1)
        ax[r, j].set_xticks([]), ax[r, j].set_yticks([])
        if j == 0:
            ax[r, j].set_ylabel(name, fontsize=6.5)
plotlib.save(f, "p6_impute")

## Slide: Imputation — ratings, when most entries are missing

200 users x 100 items, rank-3 tastes. A share of the ratings is observed, a quarter of those is
held out. Same five fillers (rank 3, random init: the best one for the sparse regime). The
fewer the observed ratings, the more the structure of the model matters.

In [ ]:
rng = np.random.default_rng(SEED)
Wt, Ht = rng.gamma(2, 1, (200, 3)), rng.gamma(2, 1, (3, 100))
Rt = np.clip(Wt @ Ht / (Wt @ Ht).max() * 5 + 0.3 * rng.normal(size=(200, 100)), 0.5, 5)
rows = []
for share in (0.5, 0.2, 0.1):
    obs = rng.random(Rt.shape) < share
    test = obs & (rng.random(Rt.shape) < 0.25)
    train = obs & ~test
    Rm = np.where(train, Rt, np.nan)
    mu = np.nanmean(Rm, 0)
    mu = np.where(np.isnan(mu), np.nanmean(Rm), mu)
    hid = lambda Y: float(np.sqrt(((Y - Rt)[test] ** 2).mean()))  # noqa: E731
    Xi = np.where(train, Rt, mu)
    for _ in range(60):
        p = PCA(3).fit(Xi)
        Xi = np.where(train, Rt, p.inverse_transform(p.transform(Xi)))
    row = {"observed": share, "item mean": hid(np.tile(mu, (200, 1))), "PCA (iterative)": hid(Xi)}
    for name in SOLVERS:
        row[name] = hid(nmf_fill(name, Rm, 3, "random"))
    rows.append(row)
slide("Imputation of ratings: RMSE on held-out ratings (rank 3)")
cf = pl.DataFrame(rows)
print(cf.with_columns(pl.col(pl.Float64).round(3)))
sparse = cf.filter(pl.col("observed") == 0.1).row(0, named=True)
dense = cf.filter(pl.col("observed") == 0.5).row(0, named=True)
best_n = min(sparse[k] for k in SOLVERS)
RESULTS["imputation, sparse ratings: 1 / RMSE"] = (
    1 / sparse["PCA (iterative)"],
    1 / sparse["PCA (iterative)"],
    1 / best_n,
)
assert best_n < sparse["PCA (iterative)"] and best_n < sparse["item mean"], "sparse: NMF fillers beat iterative PCA"
assert dense["PCA (iterative)"] <= min(dense[k] for k in SOLVERS) + 0.01, (
    "dense: iterative PCA is as good as any NMF filler"
)
slide("Which solver fills best?")
print(
    {k: round(sparse[k], 3) for k in SOLVERS}, "at 10 % observed;", {k: round(dense[k], 3) for k in SOLVERS}, "at 50 %"
)

f, ax = plt.subplots(figsize=(plotlib.COL, 1.5))
shares = cf["observed"].to_list()
for name, col, ls in (
    ("PCA (iterative)", BLUE, "-"),
    ("HALS", ORANGE, "-"),
    ("rwnmf", GREEN, "-"),
    ("MU", RED, "--"),
    ("item mean", GREY, ":"),
):
    ax.plot(shares, cf[name], color=col, ls=ls, lw=1.2, marker="o", ms=2.5, label=name)
ax.set_xlabel("share of ratings observed"), ax.set_ylabel("held-out RMSE")
ax.set_xticks(shares), ax.invert_xaxis()
ax.legend(fontsize=5.5, ncol=2, loc="upper left")
plotlib.save(f, "p6_impute_sparse")

## Slide: Scoreboard — who wins where

The key metric of every use case in one table (higher is better; nan = not applicable:
NMF needs non-negative data, soft memberships and blink removal need other structure).

In [ ]:
tab = pl.DataFrame([{"use case": k, "PCA": v[0], "ICA": v[1], "NMF": v[2]} for k, v in RESULTS.items()])
slide("Scoreboard of the use cases")
print(tab.with_columns(pl.col(pl.Float64).round(3)))

In [ ]:
LABELS = {
    "sound: spectrum corr": "sound spectra",
    "strokes: Hoyer sparseness": "digit strokes",
    "denoise c=16: 1 - rel. error": "denoising",
    "features c=16: MCC": "features (MCC)",
    "blink removal: 1 / RMSE": "blink removal",
    "anomaly: AUC": "anomaly (AUC)",
    "imputation, dense pixels: 1 / RMSE": "imputation, pixels",
    "imputation, sparse ratings: 1 / RMSE": "imputation, sparse",
}
METHODS = ["PCA", "ICA", "NMF"]
board = {}
for key, name in LABELS.items():
    vals = dict(zip(METHODS, RESULTS[key]))
    best = max(v for v in vals.values() if np.isfinite(v))
    board.update({(name, m): v / best for m, v in vals.items() if np.isfinite(v)})  # relative to the best: 1.0 wins
f = plotlib.score_board(board, list(LABELS.values()), METHODS, {"PCA": BLUE, "ICA": GREEN, "NMF": ORANGE}, left=0.2)
plotlib.save(f, "p6_scoreboard")
slide("Scoreboard, relative to the best method (1.0)")
for (name, m), v in board.items():
    if v == 1.0:
        print(f"{name}: {m}")